In [1]:
from pathlib import Path
import pandas as pd
import numpy as np

BASE = Path(r"C:\Users\ghkdr\OneDrive\바탕 화면\이스트캠프\데이콘\credit_default")
DATA_DIR = BASE / "data"
FE_DIR = DATA_DIR / "fe"

train = pd.read_csv(FE_DIR / "train_fe_v3_clean.csv")
test  = pd.read_csv(FE_DIR / "test_fe_v3_clean.csv")

print(train.shape, test.shape)
train.head()

(26451, 23) (10000, 22)


,gender,car,reality,child_num,income_total,income_type,edu_type,family_type,house_type,DAYS_BIRTH,...,email,occyp_type,family_size,begin_month,before_EMPLOYED,Age,EMPLOYED,income_per_family,ability,credit
0,F,N,N,0,12.218500,Commercial associate,Higher education,Married,Municipal apartment,13899,...,0,NaN,2.0,6.0,9190,38,12,101250.0,10.882416,1.0
1,F,N,Y,1,12.419170,Commercial associate,Secondary / secondary special,Civil marriage,House / apartment,11380,...,1,Laborers,3.0,5.0,9840,31,4,82500.0,19.156347,1.0
2,M,Y,Y,0,13.017005,Working,Higher education,Married,House / apartment,19087,...,0,Managers,2.0,22.0,14653,52,12,225000.0,19.131840,2.0
3,F,N,Y,0,12.218500,Commercial associate,Secondary / secondary special,Married,House / apartment,15088,...,0,Sales staff,2.0,37.0,12996,41,5,101250.0,11.786962,0.0
4,F,Y,Y,0,11.967187,State servant,Higher education,Married,House / apartment,15037,...,0,Managers,2.0,26.0,12932,41,5,78750.0,9.187959,2.0


In [2]:
y = train["credit"]
X = train.drop(columns=["credit"])
X_test = test.copy()

print(X.shape, y.shape)

(26451, 22) (26451,)


In [3]:
print(X.dtypes.value_counts())

int64      9
str        8
float64    5
Name: count, dtype: int64


In [4]:
# 범주형 컬럼(문자열) 추출
cat_cols = X.select_dtypes(include=["object", "string"]).columns.tolist()
num_cols = [c for c in X.columns if c not in cat_cols]

print("cat_cols:", len(cat_cols), cat_cols)
print("num_cols:", len(num_cols), num_cols)

# CatBoost는 column index로도 받으니까 인덱스도 만들어두자
cat_idx = [X.columns.get_loc(c) for c in cat_cols]
print("cat_idx:", cat_idx)

cat_cols: 8 ['gender', 'car', 'reality', 'income_type', 'edu_type', 'family_type', 'house_type', 'occyp_type']
num_cols: 14 ['child_num', 'income_total', 'DAYS_BIRTH', 'DAYS_EMPLOYED', 'work_phone', 'phone', 'email', 'family_size', 'begin_month', 'before_EMPLOYED', 'Age', 'EMPLOYED', 'income_per_family', 'ability']
cat_idx: [0, 1, 2, 5, 6, 7, 8, 14]


In [5]:
cat_cols = X.select_dtypes(include=["object", "string"]).columns.tolist()

X[cat_cols] = X[cat_cols].astype("string")
X[cat_cols] = X[cat_cols].fillna("__MISSING__")


In [6]:
cat_cols = X.select_dtypes(include=["object", "string"]).columns.tolist()

print("cat_cols:", len(cat_cols))
print("NaN in cat cols:", X[cat_cols].isna().sum().sum())
print("Any NaN in whole X:", X.isna().sum().sum())

cat_cols: 8
NaN in cat cols: 0
Any NaN in whole X: 0


In [7]:
num_cols = X.columns.difference(cat_cols).tolist()
num_nan = X[num_cols].isna().sum().sum()
print("NaN in numeric cols:", num_nan)

if num_nan > 0:
    X[num_cols] = X[num_cols].fillna(X[num_cols].median(numeric_only=True))

NaN in numeric cols: 0


In [8]:
import numpy as np
import pandas as pd
from catboost import Pool, CatBoostClassifier
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import log_loss

# 1) y 형태 통일 (CatBoost/Logloss 안정화)
y = pd.Series(y).astype(int)

# 2) cat_cols가 "컬럼명" 리스트인지 확인 후 -> 인덱스로 변환 (가장 안전)
cat_features = [X.columns.get_loc(c) for c in cat_cols]  # cat_cols가 컬럼명 리스트라고 가정

# 3) 카테고리 컬럼은 무조건 문자열로 (NaN/float 혼입 방지)
for c in cat_cols:
    X[c] = X[c].astype("string")

# 4) 전체 NaN 체크
print("X shape:", X.shape, " y shape:", y.shape)
print("Any NaN in X:", X.isna().any().any())
print("Any NaN in y:", y.isna().any())
print("cat features count:", len(cat_features))

X shape: (26451, 22)  y shape: (26451,)
Any NaN in X: False
Any NaN in y: False
cat features count: 8


In [9]:
print("X shape:", X.shape, "X_test shape:", X_test.shape)
print("Only in X:", sorted(set(X.columns) - set(X_test.columns)))
print("Only in X_test:", sorted(set(X_test.columns) - set(X.columns)))

X shape: (26451, 22) X_test shape: (10000, 22)
Only in X: []
Only in X_test: []


In [16]:
X["occyp_count"] = X["occyp_type"].map(X["occyp_type"].value_counts())
X["income_type_count"] = X["income_type"].map(X["income_type"].value_counts())

X_test["occyp_count"] = X_test["occyp_type"].map(X["occyp_type"].value_counts())
X_test["income_type_count"] = X_test["income_type"].map(X["income_type"].value_counts())

In [17]:
import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedKFold
from catboost import CatBoostClassifier, Pool

SEED = 42

# 0) 안전장치: X/X_test는 반드시 "pandas DataFrame" 이어야 함 (numpy면 컬럼명 사라짐)
assert isinstance(X, pd.DataFrame)
assert isinstance(X_test, pd.DataFrame)

# 1) test 컬럼 순서 고정 (혹시 모를 reorder 방지)
X_test = X_test.reindex(columns=X.columns)

# 2) cat_cols는 "컬럼명"으로만 (인덱스 리스트 사용 금지)
cat_cols = X.select_dtypes(include=["object", "string"]).columns.tolist()

# 3) 범주형은 전부 문자열로 고정 (CatBoost가 제일 안정적으로 먹음)
for c in cat_cols:
    X[c] = X[c].astype("string").fillna("NA")
    X_test[c] = X_test[c].astype("string").fillna("NA")

n_class = y.nunique()
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)

test_pred = np.zeros((len(X_test), n_class))
oof = np.zeros((len(X), n_class))

for fold, (tr_idx, va_idx) in enumerate(skf.split(X, y), 1):
    X_tr, X_va = X.iloc[tr_idx], X.iloc[va_idx]
    y_tr, y_va = y.iloc[tr_idx], y.iloc[va_idx]

    train_pool = Pool(X_tr, y_tr, cat_features=cat_cols)
    valid_pool = Pool(X_va, y_va, cat_features=cat_cols)
    test_pool  = Pool(X_test,     cat_features=cat_cols)

    model = CatBoostClassifier(
        loss_function="MultiClass",
        eval_metric="MultiClass",
        iterations=5000,
        learning_rate=0.03,
        depth=8,
        random_seed=SEED,
        od_type="Iter",
        od_wait=200,
        use_best_model=True,
        verbose=200
    )

    model.fit(train_pool, eval_set=valid_pool)

    oof[va_idx] = model.predict_proba(X_va)
    test_pred += model.predict_proba(test_pool) / skf.n_splits

# 4) 제출 파일 생성 (sample_submission 형식에 맞게 수정)


0:	learn: 1.0812720	test: 1.0812553	best: 1.0812553 (0)	total: 50.9ms	remaining: 4m 14s
200:	learn: 0.7845080	test: 0.7987464	best: 0.7987464 (200)	total: 9.05s	remaining: 3m 36s
400:	learn: 0.7464886	test: 0.7858358	best: 0.7858358 (400)	total: 19.1s	remaining: 3m 39s
600:	learn: 0.7100864	test: 0.7742108	best: 0.7742108 (600)	total: 29.2s	remaining: 3m 34s
800:	learn: 0.6774424	test: 0.7646404	best: 0.7646404 (800)	total: 39.2s	remaining: 3m 25s
1000:	learn: 0.6488977	test: 0.7570238	best: 0.7570066 (999)	total: 49.3s	remaining: 3m 16s
1200:	learn: 0.6230612	test: 0.7522626	best: 0.7522626 (1200)	total: 59.5s	remaining: 3m 8s
1400:	learn: 0.6007958	test: 0.7476212	best: 0.7476212 (1400)	total: 1m 9s	remaining: 2m 58s
1600:	learn: 0.5780749	test: 0.7438894	best: 0.7438894 (1600)	total: 1m 19s	remaining: 2m 49s
1800:	learn: 0.5568352	test: 0.7402057	best: 0.7402057 (1800)	total: 1m 29s	remaining: 2m 39s
2000:	learn: 0.5374318	test: 0.7371031	best: 0.7370754 (1999)	total: 1m 39s	remaini

In [18]:
# 예측 확률 컬럼들만 float로 강제
pred_cols = sub.columns[1:]          # 첫 컬럼은 id/index일 가능성이 큼
sub[pred_cols] = sub[pred_cols].astype("float64")

# 대입
sub.loc[:, pred_cols] = test_pred

save_path = BASE / "submissions" / "submission_cv5_catboost.csv"
sub.to_csv(save_path, index=False)
print("Saved:", save_path)
print(sub.head())

Saved: C:\Users\ghkdr\OneDrive\바탕 화면\이스트캠프\데이콘\credit_default\submissions\submission_cv5_catboost.csv
   index         0         1         2
0  26457  0.036702  0.031576  0.931722
1  26458  0.166440  0.169077  0.664483
2  26459  0.077267  0.104071  0.818662
3  26460  0.103428  0.110799  0.785772
4  26461  0.061842  0.156037  0.782121


In [ ]:
print(type(cat_features[0]), cat_features[:5])

In [ ]:
print(y.dtype, np.unique(y))

In [ ]:
X.isna().any().any()

In [ ]:
for df in [X]:
    df["begin_year"] = df["begin_month"] // 12
    df["begin_mod12"] = df["begin_month"] % 12

In [ ]:
X["occyp_edu"] = X["occyp_type"].astype(str) + "_" + X["edu_type"].astype(str)
X["income_family"] = X["income_type"].astype(str) + "_" + X["family_type"].astype(str)
X["car_reality"] = X["car"].astype(str) + "_" + X["reality"].astype(str)

In [ ]:
# 1. cat_cols 재정의
cat_cols = X.select_dtypes(include=["object", "string"]).columns.tolist()

# 2. 다시 split
from sklearn.model_selection import train_test_split
from catboost import Pool

X_train, X_valid, y_train, y_valid = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

# 3. Pool 다시 생성
train_pool = Pool(X_train, y_train, cat_features=cat_cols)
valid_pool = Pool(X_valid, y_valid, cat_features=cat_cols)

# 4. 학습
model.fit(train_pool, eval_set=valid_pool)

In [ ]:
from sklearn.metrics import log_loss
import numpy as np

valid_proba = model.predict_proba(X_valid)

print("valid logloss:", log_loss(y_valid, valid_proba))

In [ ]:
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import log_loss
from catboost import CatBoostClassifier, Pool
import numpy as np

# cat_features 인덱스 버전
cat_cols = X.select_dtypes(include=["object", "string"]).columns.tolist()
cat_features = [X.columns.get_loc(c) for c in cat_cols]

SEED = 42
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)

oof = np.zeros((len(X), y.nunique()), dtype=float)

for fold, (tr_idx, va_idx) in enumerate(skf.split(X, y), 1):
    print(f"Fold {fold}")

    X_tr, X_va = X.iloc[tr_idx], X.iloc[va_idx]
    y_tr, y_va = y.iloc[tr_idx], y.iloc[va_idx]

    train_pool = Pool(X_tr, y_tr, cat_features=cat_features)
    valid_pool = Pool(X_va, y_va, cat_features=cat_features)

    model = CatBoostClassifier(
        loss_function="MultiClass",
        eval_metric="MultiClass",
        iterations=3000,          # 5000은 오래 걸릴 수 있음
        learning_rate=0.03,
        depth=8,
        random_seed=SEED,
        early_stopping_rounds=200,
        verbose=200
    )

    model.fit(train_pool, eval_set=valid_pool)
    oof[va_idx] = model.predict_proba(X_va)

cv_score = log_loss(y, oof)
print("5-Fold CV logloss:", cv_score)